In [ ]:
!pip install scikit-fuzzy umap-learn matplotlib numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import skfuzzy as fuzz
import umap

warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/semantic-search-system'
os.makedirs(f'{BASE}/experiments', exist_ok=True)

embeddings = np.load(f'{BASE}/models/embeddings.npy')
with open(f'{BASE}/data/processed/clean_corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

print(f'Embeddings shape : {embeddings.shape}')
print(f'Corpus size      : {len(corpus["texts"])} documents')

In [ ]:
print('Running UMAP — takes 3-8 minutes...')

reducer_2d = umap.UMAP(n_components=2,  n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
reducer_nd = umap.UMAP(n_components=20, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)

emb_2d = reducer_2d.fit_transform(embeddings)
emb_nd = reducer_nd.fit_transform(embeddings)

print(f'2D shape : {emb_2d.shape}')
print(f'20D shape: {emb_nd.shape}')

In [ ]:
N_CLUSTERS = 15
print(f'Running Fuzzy C-Means with {N_CLUSTERS} clusters...')

cntr, membership_matrix, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    emb_nd.T.astype(np.float64),
    c=N_CLUSTERS,
    m=2.0,
    error=0.005,
    maxiter=1000,
    init=None,
    seed=42
)

print(f'FPC score             : {fpc:.4f}  (1.0=perfect, 1/c=random)')
print(f'Membership matrix     : {membership_matrix.shape}')

In [ ]:
hard_labels        = np.argmax(membership_matrix, axis=0)
dominant_membership = np.max(membership_matrix, axis=0)

print(f'{"Cluster":>8}  {"Docs":>8}  {"Avg Membership":>15}')
print('-' * 38)
unique, counts = np.unique(hard_labels, return_counts=True)
for c, n in zip(unique, counts):
    avg = membership_matrix[c][hard_labels == c].mean()
    print(f'Cluster {c:2d}  {n:8d}  {avg:15.3f}')

In [ ]:
BOUNDARY_THRESHOLD = 0.40
boundary_mask = dominant_membership < BOUNDARY_THRESHOLD

print(f'Boundary docs  : {boundary_mask.sum()}')
print(f'Core docs      : {(~boundary_mask).sum()}')

boundary_indices = np.where(boundary_mask)[0][:3]
print('\nSample boundary documents:')
for idx in boundary_indices:
    top2 = np.argsort(membership_matrix[:, idx])[-2:][::-1]
    print(f'\n  Doc {idx} [{corpus["categories"][idx]}]')
    print(f'    Cluster {top2[0]}: {membership_matrix[top2[0], idx]:.3f}')
    print(f'    Cluster {top2[1]}: {membership_matrix[top2[1], idx]:.3f}')
    print(f'    Text: {corpus["texts"][idx][:100]}...')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Fuzzy C-Means Clustering on 20 Newsgroups', fontsize=14, fontweight='bold')

colors = cm.tab20(np.linspace(0, 1, N_CLUSTERS))
for c in range(N_CLUSTERS):
    mask = hard_labels == c
    axes[0].scatter(emb_2d[mask,0], emb_2d[mask,1], s=5, alpha=0.5,
                    color=colors[c], label=f'C{c}')
axes[0].set_title('Cluster Assignments (UMAP 2D)')
axes[0].legend(fontsize=6, ncol=3)

sc = axes[1].scatter(emb_2d[:,0], emb_2d[:,1],
                     c=dominant_membership, s=5, cmap='RdYlGn', alpha=0.7)
plt.colorbar(sc, ax=axes[1], label='Membership certainty')
axes[1].set_title('Membership Certainty (Green=core, Red=boundary)')

plt.tight_layout()
plt.savefig(f'{BASE}/experiments/cluster_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/cluster_visualization.png')

In [ ]:
cluster_model = {
    'cntr'      : cntr,
    'n_clusters': N_CLUSTERS,
    'fpc'       : fpc,
    'emb_2d'    : emb_2d,
    'emb_nd'    : emb_nd,
    'reducer_2d': reducer_2d,
    'reducer_nd': reducer_nd
}

with open(f'{BASE}/models/cluster_model.pkl', 'wb') as f:
    pickle.dump(cluster_model, f)

np.save(f'{BASE}/models/membership_matrix.npy', membership_matrix)

print('Saved: models/cluster_model.pkl')
print('Saved: models/membership_matrix.npy')